In [ ]:
# In langgraph,nodes typically execute in a sequence defined by edges
# but when tasks don't depend on each other output, you can run them 
# in parallel. This is achieved by :
# a) Defining multiple nodes that can operate independantly 
# b) Connecting them to common starting point.
# c) Merging their outputs to a downstream node if needed

import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model = "openai/gpt-oss-20b")


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display

class State(TypedDict):
    topic: str
    characters: str
    settings: str
    premise: str
    story_intro: str

In [ ]:
def generate_character(state: State):
    """Generate character description"""
    msg = llm.invoke(f"Create two character names and brief traits for a story about {state['topic']}")
    return {"characters": msg.content}

def generate_settings(state: State):
    """Generate a story setting"""
    msg = llm.invoke(f"Describe a vivid setting for a story about {state['topic']}")
    return {"settings": msg.content}

def generate_premise(state: State):
    """Generate a story premise"""
    msg = llm.invoke(f"Write one sentence plot for a story about {state['topic']}")
    return {"premise": msg.content}

def combine_elements(state: State):
    """Combine characters, settings and premise into an intro"""
    msg = llm.invoke(
        f"Write a short introduction to these story elements:\n"
        f"Characters: {state['characters']}\n"
        f"Settings: {state['settings']}\n"
        f"Premise: {state['premise']}\n"
    )
    return {"story_intro": msg.content}

In [ ]:
graph = StateGraph(State)
graph.add_node("character", generate_character)
graph.add_node("settings", generate_settings)
graph.add_node("premise", generate_premise)
graph.add_node("combine", combine_elements)

graph.add_edge(START, "character")
graph.add_edge(START, "settings")
graph.add_edge(START, "premise")
graph.add_edge("character", "combine")
graph.add_edge("settings", "combine")
graph.add_edge("premise", "combine")
graph.add_edge("combine", END)

compiled_graph = graph.compile()
graph_image = compiled_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))

In [ ]:
state = {"topic":"TimeTravel"}
result = compiled_graph.invoke(state)
print(result["story_intro"])